# **ReAct Agent in LangGraph**

ReAct = **Re**ason + **Act**.
The LLM reasons about whether it needs a tool, calls it, gets the result, reasons again — looping until no more tool calls are needed.

Graph structure:
```
START → llm_node ──(tool calls?)──► tool_node → llm_node ...
                 └──(no calls) ──► END
```

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, ToolMessage
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os

load_dotenv()

if os.environ.get('GROQ_API_KEY'):
    print("Groq API Key is set.")
else:
    raise ValueError("GROQ_API_KEY not found.")

In [ ]:
llm = ChatGroq(model="llama3-8b-8192", temperature=0)

### **Tools**

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def search_duckduckgo(query: str) -> str:
    """This tool searches the latest news on DuckDuckGo for the given query and returns the results."""
    duck_search = DuckDuckGoSearchRun()
    return duck_search.invoke(query)

In [ ]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

@tool
def arxiv_tool(query: str) -> str:
    """This tool allows you to query the arXiv database for research papers."""
    arxiv_query = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())
    return arxiv_query.invoke(query)

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

@tool
def wiki_tool(query: str):
    """This tool allows you to search Wikipedia for information on a given topic."""
    wiki_query = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki_query.invoke(query)

In [ ]:
@tool
def personal_info(name: str):
    """Use this tool to get personal information about Alice, Bob, or Charlie."""
    info = {
        "Alice":   "Alice is a software engineer with 5 years of experience in AI.",
        "Bob":     "Bob is a data scientist who loves working with large datasets.",
        "Charlie": "Charlie is a product manager with a background in tech startups."
    }
    return info.get(name, "No information available for this person.")

### **Tool Binding**

In [ ]:
tools = [search_duckduckgo, arxiv_tool, wiki_tool, personal_info]

# Attach tool schemas to the LLM so it knows what tools exist
llm_with_tools = llm.bind_tools(tools)

### **State Schema**

In [ ]:
from typing import TypedDict, List

class graph_schema(TypedDict):
    messages: List   # running list of HumanMessage / AIMessage / ToolMessage objects

### **Node Functions**

In [ ]:
def llm_node(state: graph_schema) -> graph_schema:

    messages = state['messages']

    # System prompt tells the LLM it can use tools; {input} is the conversation so far
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant that can use tools to answer questions."),
        ("human",  "{input}")
    ])

    # Chain the prompt template → LLM-with-tools
    chain = prompt | llm_with_tools

    # The LLM returns an AIMessage that may contain .tool_calls (if it wants to call a tool)
    response = chain.invoke({"input": messages})

    # Append the LLM's reply (with potential tool_calls) to the message history
    state['messages'] = messages + [response]
    return state

In [ ]:
def tool_node(state: graph_schema) -> graph_schema:
    """Execute every tool call the LLM requested and append the results as ToolMessages."""

    messages = state['messages']

    # Build a name→tool lookup so we can dispatch by name
    tools_by_name = {tool.name: tool for tool in tools}

    tool_results = []

    # The last message is the AIMessage that contains tool_calls
    for tool_call in messages[-1].tool_calls:

        tool    = tools_by_name[tool_call["name"]]         # look up the tool by name
        result  = tool.invoke(tool_call["args"])           # run the tool

        # ToolMessage wraps the result and links it back to the specific call via tool_call_id
        tool_results.append(
            ToolMessage(content=result, tool_call_id=tool_call["id"])
        )

    # Append all tool results to the history; llm_node will process them next
    state['messages'] = messages + tool_results
    return state

### **Conditional Edge Function**
This decides where to go after `llm_node`: if the LLM made tool calls → `tool_node`; otherwise → END.

In [ ]:
def if_tool_call(state: graph_schema) -> str:

    last_message = state['messages'][-1]

    # AIMessage has .tool_calls if the LLM wants to invoke tools; it's empty otherwise
    if last_message.tool_calls:
        return "tool_node"   # route to tool execution
    else:
        return "end"         # no more tool calls — we're done

### **Build the Graph**

In [ ]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image

graph = StateGraph(graph_schema)

graph.add_node("llm_node",  llm_node)
graph.add_node("tool_node", tool_node)

graph.add_edge(START, "llm_node")

# Conditional edge: after llm_node, call if_tool_call() to decide the next step
graph.add_conditional_edges(
    "llm_node",
    if_tool_call,
    {"tool_node": "tool_node", "end": END}   # map return values → target nodes
)

graph.add_edge("tool_node", "llm_node")   # after tool execution, go back to LLM
graph.add_edge("llm_node", END)           # llm_node can also directly reach END

react_graph = graph.compile()
Image(react_graph.get_graph().draw_mermaid_png())

### **Graph Invocation**

In [ ]:
result = react_graph.invoke({"messages": [HumanMessage(content="What is the latest news on AI?")]})

# Print each message in the conversation (Human → AI (with tool calls) → Tool results → AI final)
for msg in result['messages']:
    msg.pretty_print()

In [ ]:
# stream() yields updates after each node runs — good for watching the ReAct loop in real time
for chunk in react_graph.stream(
    {"messages": [HumanMessage(content="Who is Alice?")]},
    stream_mode="updates"
):
    print(chunk)